In [56]:
import os
from typing import List
from pydantic import BaseModel, Field
from langchain_core.documents import Document
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS 
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langgraph.graph import StateGraph, END

In [57]:
from dotenv import load_dotenv
load_dotenv(override=True)

class RAGCoTState(BaseModel):
    question: str
    sub_steps: List[str] = Field(default_factory=list)
    retrieved_docs: List[Document] = Field(default_factory=list)
    answer: str = ""

llm = ChatOpenAI(model="gpt-4o-mini")

In [58]:
docs = TextLoader("piyush.txt", encoding="utf-8").load()
splitter = RecursiveCharacterTextSplitter(separators=["\n"], chunk_size=500, chunk_overlap=50)
split_docs = splitter.split_documents(docs)
split_docs

[Document(metadata={'source': 'piyush.txt'}, page_content='Contact\npiyushthakur3613@gmail.com\nwww.linkedin.com/in/\npiyushthakurthedatagenius\n(LinkedIn)\nTop Skills\nData Analysis\nLLM\nArtificial Intelligence (AI)\nCertifications\nExploratory Data Analysis in SQL\nWriting Efficient Python Code\ncleaning data in python\nFOUNDATION DATA ANALYTICS\nsupervised learning with scikitlearn\nPiyush Thakurr\nExperienced Data Scientist |Expert in GenAI, LLM, Agentic AI |\nMachine Learning, NLP, DL, Statistical Analysis, and Data Mining'),
 Document(metadata={'source': 'piyush.txt'}, page_content='| Skilled in Python, R, and SQL | Big Data Technologies (Hadoop,\nSpark)|Marketing Analytics\nUnited States\nSummary\nseasoned Data Scientist/AI Engineer  with over 8+ years\nof professional experience, specializing in Data science,\nPython Development ,Machine Learning, Data Mining ,Data\nanalysis ,Business Development and Statistical Analysis. Throughout\nmy career, I have demonstrated expertise in

In [59]:
from langchain_openai import OpenAIEmbeddings as OAIEmbed
embeddings = OAIEmbed(model="text-embedding-3-small")
vectordb = FAISS.from_documents(split_docs, embeddings)
retriever = vectordb.as_retriever(search_kwargs={"k": 3})

In [60]:
def plan_steps(state: RAGCoTState) -> RAGCoTState:
    prompt = f"""You are a helpful assistant for question decomposition.
    Based on the question, generate a list of subquestions that can help answer the main question.
    {state.question}"""
    result = llm.invoke(prompt).content
    subqs = [line.strip("- ").strip() for line in result.split("\n") if line.strip()]
    return state.model_copy(update={"sub_steps": subqs})

In [61]:
from langchain_openai import ChatOpenAI

def reterieve_per_Step(state: RAGCoTState)->RAGCoTState:
    retrieved_docs = []
    for sub_q in state.sub_steps:
        docs = retriever.invoke(sub_q)
        retrieved_docs.extend(docs)
    return state.model_copy(update={"retrieved_docs": retrieved_docs})


In [62]:
def generate_answer(state: RAGCoTState) -> RAGCoTState:
    prompt = f"""You are a helpful assistant for question answering.
based on the question and the retrieved documents, generate a comprehensive answer to the main question.
Question: {state.question}
Retrieved Documents: {state.retrieved_docs}
based on the above question and retrieved documents, generate a comprehensive answer to the main question."""
    answer = llm.invoke(prompt).content
    return state.model_copy(update={"answer": answer})

In [63]:
workflow= StateGraph(RAGCoTState)

workflow.add_node("planner", plan_steps)
workflow.add_node("retriever", reterieve_per_Step)
workflow.add_node("answer_generator", generate_answer)

workflow.set_entry_point("planner")
workflow.add_edge("planner", "retriever")
workflow.add_edge("retriever", "answer_generator")

workflow.add_edge("answer_generator", END)

graph=workflow.compile()

In [65]:
if __name__ == "__main__":
    state1 = RAGCoTState(question="What are the key contributions of Piyush's research?")
    final_state = graph.invoke(state1)
    print(final_state["answer"])

Piyush's research and contributions span several key areas in data science, artificial intelligence, and machine learning, demonstrating a robust proficiency in both theoretical and practical applications. Below are the main contributions highlighted in the retrieved documents:

1. **Expertise in Generative AI and Language Models**: Piyush is recognized as an expert in generative AI (GenAI) and language models (LLMs), including technologies like GPT, Langchain, and Hugging Face. This expertise enables him to leverage these frameworks to build innovative solutions, including advanced applications in natural language processing.

2. **Strong Technical Skills Across Multiple Platforms**: He possesses a diverse skill set in programming and statistical tools, including proficiency in Python (utilizing libraries such as NumPy, Pandas, Matplotlib, and Scikit-learn), R, SQL, and big data technologies like Hadoop and Spark. This range of skills allows him to handle both structured and unstructu